In [1]:
import torch
import pandas as pd
import numpy as np
import regex as re
from epsilon_transformers.persistence import Persister


/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'repr' attribute with value False was provided to the `Field()` function, which has no effect in the context it was used. 'repr' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because an `Annotated` type alias using the `type` statement was used, or if the `Field()` function was attached to a single member of a union type.
  warnings.warn(
/opt/anaconda3/envs/epstrans311/lib/python3.11/site-packages/pydantic/_internal/_generate_schema.py:2249: UnsupportedFieldAttributeWarning: The 'frozen' attribute with value True was provided to the `Field()` function, which has no effect in the context it was used. 'frozen' is field-specific metadata, and can only be attached to a model field using `Annotated` metadata or by assignment. This may have happened because a

In [3]:
def analyze_model_ranks(model):
    results=[]
    for name, param in model.named_parameters():
        tensor=param.detach()
        if tensor.dim()<2:
            continue
        if tensor.dim()==2:
            rank=torch.linalg.matrix_rank(tensor).item()
            full_rank=min(tensor.shape)
            results.append({
                "parameter": name,
                "shape": tuple(tensor.shape),
                "rank": rank,
                "full_rank": full_rank,
                "deficiency": full_rank - rank
            })
        elif tensor.dim()==4:
            n_heads=tensor.shape[0]
            for head_idx in range(n_heads):
                head_mat=tensor[head_idx]
                rank=torch.linalg.matrix_rank(head_mat).item()
                full_rank=min(head_mat.shape)
                results.append({
                    "parameter": f"{name}_head_{head_idx}",
                    "shape": tuple(head_mat.shape),
                    "rank": rank,
                    "full_rank": full_rank,
                    "deficiency": full_rank - rank
                })
    return pd.DataFrame(results)                

In [18]:
persister = Persister(save_dir="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/small checkpoints/1lyr_0.15_0.6_lr0.01adm_40M_400/epsilon-transformers/models/linrmess3_thry/1lyr_0.15_0.6_lr0.01adm_40M_400/")
model_path="/Users/sbhandari/Documents/GitHub/epsilon-transformers/models/small checkpoints/1lyr_0.15_0.6_lr0.01adm_40M_400/epsilon-transformers/models/linrmess3_thry/1lyr_0.15_0.6_lr0.01adm_40M_400/checkpoint_801_tokens_40000000.pt"
model = persister.load_model(model_path)
df_ranks = analyze_model_ranks(model)

In [22]:
from epsilon_transformers.process.processes import PROCESS_REGISTRY
from torch import device
process_name = 'Linear_Mess3'
process_params ={
    "x": 0.15,
    "a": 0.6
}
seq_len = 10
vocab = 3
if process_name in PROCESS_REGISTRY:
    process=PROCESS_REGISTRY[process_name](**process_params)
history=process.generate_process_history(total_length=10)
if torch.cuda.is_available():
    device = device("cuda:0")
elif torch.backends.mps.is_available():
    device = device("mps")
else:
    device = device("cpu")
input_seq=torch.tensor([history.symbols],dtype=torch.long,device=device)    

In [24]:
df_norms = measure_residual_norms(model, input_seq)

In [25]:
print(df_norms)

   layer_idx                     hook_name        component       norm
0          0       blocks.0.hook_resid_pre   hook_resid_pre   6.600586
1          0  blocks.0.ln1.hook_normalized  hook_normalized   1.183700
2          0        blocks.0.hook_attn_out    hook_attn_out   2.031928
3          0       blocks.0.hook_resid_mid   hook_resid_mid   7.960988
4          0  blocks.0.ln2.hook_normalized  hook_normalized   2.234404
5          0         blocks.0.mlp.hook_pre         hook_pre   5.544907
6          0        blocks.0.mlp.hook_post        hook_post   0.546068
7          0         blocks.0.hook_mlp_out     hook_mlp_out   4.530251
8          0      blocks.0.hook_resid_post  hook_resid_post  10.491097
9         -1      ln_final.hook_normalized  hook_normalized   0.432001


In [20]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

            parameter    shape  rank  full_rank  deficiency
0           embed.W_E   (3, 3)     3          3           0
1     pos_embed.W_pos  (10, 3)     3          3           0
2   blocks.0.attn.b_Q   (1, 3)     1          1           0
3   blocks.0.attn.b_K   (1, 3)     1          1           0
4   blocks.0.attn.b_V   (1, 3)     1          1           0
5   blocks.0.mlp.W_in  (3, 12)     3          3           0
6  blocks.0.mlp.W_out  (12, 3)     3          3           0
7         unembed.W_U   (3, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [5]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

             parameter      shape  rank  full_rank  deficiency
0            embed.W_E    (3, 64)     3          3           0
1      pos_embed.W_pos   (10, 64)    10         10           0
2    blocks.0.attn.b_Q     (2, 8)     2          2           0
3    blocks.0.attn.b_K     (2, 8)     2          2           0
4    blocks.0.attn.b_V     (2, 8)     2          2           0
5    blocks.0.mlp.W_in  (64, 256)    64         64           0
6   blocks.0.mlp.W_out  (256, 64)    64         64           0
7    blocks.1.attn.b_Q     (2, 8)     2          2           0
8    blocks.1.attn.b_K     (2, 8)     2          2           0
9    blocks.1.attn.b_V     (2, 8)     2          2           0
10   blocks.1.mlp.W_in  (64, 256)    64         64           0
11  blocks.1.mlp.W_out  (256, 64)    64         64           0
12         unembed.W_U    (64, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [8]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

             parameter      shape  rank  full_rank  deficiency
0            embed.W_E    (3, 64)     3          3           0
1      pos_embed.W_pos   (10, 64)    10         10           0
2    blocks.0.attn.b_Q     (2, 8)     2          2           0
3    blocks.0.attn.b_K     (2, 8)     2          2           0
4    blocks.0.attn.b_V     (2, 8)     2          2           0
5    blocks.0.mlp.W_in  (64, 256)    64         64           0
6   blocks.0.mlp.W_out  (256, 64)    64         64           0
7    blocks.1.attn.b_Q     (2, 8)     2          2           0
8    blocks.1.attn.b_K     (2, 8)     2          2           0
9    blocks.1.attn.b_V     (2, 8)     2          2           0
10   blocks.1.mlp.W_in  (64, 256)    64         64           0
11  blocks.1.mlp.W_out  (256, 64)    64         64           0
12         unembed.W_U    (64, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [6]:
print(df_ranks)
print("\nLow Rank Matrices:\n", df_ranks[df_ranks["deficiency"] > 0])

             parameter      shape  rank  full_rank  deficiency
0            embed.W_E    (3, 64)     3          3           0
1      pos_embed.W_pos   (10, 64)    10         10           0
2    blocks.0.attn.b_Q     (2, 8)     2          2           0
3    blocks.0.attn.b_K     (2, 8)     2          2           0
4    blocks.0.attn.b_V     (2, 8)     2          2           0
5    blocks.0.mlp.W_in  (64, 256)    64         64           0
6   blocks.0.mlp.W_out  (256, 64)    64         64           0
7    blocks.1.attn.b_Q     (2, 8)     2          2           0
8    blocks.1.attn.b_K     (2, 8)     2          2           0
9    blocks.1.attn.b_V     (2, 8)     2          2           0
10   blocks.1.mlp.W_in  (64, 256)    64         64           0
11  blocks.1.mlp.W_out  (256, 64)    64         64           0
12         unembed.W_U    (64, 3)     3          3           0

Low Rank Matrices:
 Empty DataFrame
Columns: [parameter, shape, rank, full_rank, deficiency]
Index: []


In [23]:
def measure_residual_norms(model, input_seq,layer_idx=None):
    def filter(name):
        return any(name.endswith(suffix) for suffix in ['out','_normalized','_pre','_post','_mid'])
    with torch.no_grad():
        _,cache=model.run_with_cache(input_seq,names_filter=filter)
    norms=[]
    for hook_name, act in cache.items():
        if act.dim()>=2:
            norm_val=act.flatten(start_dim=2).norm(dim=-1).mean().item()
        else:
            norm_val=act.norm().item()
        match=re.search(r'blocks\.(\d+)\.', hook_name)
        layer_idx=int(match.group(1)) if match else -1
        component=hook_name.split('.')[-1]
        norms.append({
            "layer_idx": layer_idx,
            "hook_name": hook_name,
            "component": component,
            "norm": norm_val  })
    return pd.DataFrame(norms)

In [14]:
def ablation(model, input_seq,hook_name):
    def zero_hook(activations,hook):
        return torch.zeros_like(activations)
    with model.hooks(fwd_hooks=[(hook_name,zero_hook)]):
        logits=model(input_seq)
    return logits

In [17]:
input_seq=torch.tensor([[1,1,0,2,1,1,1,2,2,2]],dtype=torch.long)

In [14]:
with torch.no_grad():
    _, cache=model.run_with_cache(input_seq)
hook_name='blocks.0.hook_mlp_out'    
cache_act=cache[hook_name]
print(cache_act)
norm=cache_act.norm().item()
print(f"Norm of activations at {hook_name}: {norm}")
variance=cache_act.var().item()
print(f"Variance of activations at {hook_name}: {variance}")

tensor([[[-2.7619,  0.4948, -0.1775,  2.5151, -1.5817,  1.9888,  0.1695,
           1.6366, -1.9397,  0.0229,  3.1662, -0.6829, -1.8779,  1.3264,
          -1.2645,  0.2821,  1.8683, -0.6908,  1.4961,  0.1264, -1.0220,
           2.4129,  2.7328, -0.1900,  0.9680, -0.0261,  0.2288, -0.0255,
           0.3880,  0.2383, -1.2034, -0.2064, -0.9974, -1.4505,  0.3415,
          -2.1154, -0.6625, -0.9574, -0.1475, -0.4544,  0.9122,  0.9663,
          -0.2540,  0.6001,  2.9935,  0.8185,  0.7452,  1.8387,  1.0216,
           1.2007,  2.2094, -0.2765, -0.4427, -0.1249, -0.5228, -1.1856,
          -0.3545,  1.5152, -0.3307, -0.4498,  0.8130, -0.7449, -1.0524,
          -0.2044]]])
Norm of activations at blocks.0.hook_mlp_out: 10.361372947692871
Variance of activations at blocks.0.hook_mlp_out: 1.6703804731369019


In [19]:
with torch.no_grad():
    _, cache=model.run_with_cache(input_seq)
hook_name='blocks.0.hook_mlp_out'    
cache_act=cache[hook_name]
print(cache_act)
norm=cache_act.norm().item()
print(f"Norm of activations at {hook_name}: {norm}")
variance=cache_act.var().item()
print(f"Variance of activations at {hook_name}: {variance}")

tensor([[[-1.3950, -1.4883,  2.5038],
         [-1.7147, -2.9241,  7.0138],
         [-1.3950, -1.4883,  2.5038],
         [-0.6574, -6.1377,  5.8825],
         [-1.3950, -1.4883,  2.5038],
         [-1.3950, -1.4883,  2.5038],
         [-1.3950, -1.4883,  2.5038],
         [-0.6096, -7.0399,  7.1422],
         [-0.6165, -6.9003,  6.9438],
         [-0.6255, -6.7216,  6.6918]]])
Norm of activations at blocks.0.hook_mlp_out: 21.749683380126953
Variance of activations at blocks.0.hook_mlp_out: 16.306612014770508


tensor([[[-1.3615,  1.5457, -0.8380,  0.2007,  0.4802, -0.0870, -1.6374,
           2.1768,  0.6627,  1.5227,  0.7640, -0.4300, -1.3383,  0.1751,
          -0.8854, -1.7988,  0.6675, -0.2757,  1.1209, -0.6538, -0.5206,
          -0.0750,  0.9160, -0.8097, -0.2248, -1.5375,  0.6916,  0.9069,
           0.6678,  0.3281,  0.6525,  0.5110, -2.0519,  1.5188, -1.1874,
          -2.4656, -0.6901,  0.0657, -1.0311,  1.0094,  0.9483,  0.3592,
          -0.0220,  1.0619, -2.5305,  2.0560,  1.0988,  1.7447, -1.7507,
           2.5790, -2.1695, -1.6673, -1.0942,  0.1840, -0.1283, -1.5977,
           0.2466, -1.0082, -1.1547, -0.8117, -1.5243, -1.2212,  3.0809,
          -1.2129]]])
Norm of activations at blocks.1.hook_mlp_out: 10.168623924255371
Variance of activations at blocks.1.hook_mlp_out: 1.6260031461715698


In [16]:
with torch.no_grad():
    _, cache=model.run_with_cache(input_seq)
hook_name_1='blocks.1.hook_mlp_out'    
cache_act_1=cache[hook_name_1]
hook_name_0='blocks.0.hook_mlp_out'
cache_act=cache[hook_name_0]
diff=cache_act_1 - cache_act
print(cache_act)
norm=diff.norm().item()
print(f"Norm of difference in activations between {hook_name_1} and {hook_name_0}: {norm}")
variance=diff.var().item()
print(f"Variance of difference in activations between {hook_name_1} and {hook_name_0}: {variance}")

tensor([[[-2.7619,  0.4948, -0.1775,  2.5151, -1.5817,  1.9888,  0.1695,
           1.6366, -1.9397,  0.0229,  3.1662, -0.6829, -1.8779,  1.3264,
          -1.2645,  0.2821,  1.8683, -0.6908,  1.4961,  0.1264, -1.0220,
           2.4129,  2.7328, -0.1900,  0.9680, -0.0261,  0.2288, -0.0255,
           0.3880,  0.2383, -1.2034, -0.2064, -0.9974, -1.4505,  0.3415,
          -2.1154, -0.6625, -0.9574, -0.1475, -0.4544,  0.9122,  0.9663,
          -0.2540,  0.6001,  2.9935,  0.8185,  0.7452,  1.8387,  1.0216,
           1.2007,  2.2094, -0.2765, -0.4427, -0.1249, -0.5228, -1.1856,
          -0.3545,  1.5152, -0.3307, -0.4498,  0.8130, -0.7449, -1.0524,
          -0.2044]]])
Norm of difference in activations between blocks.1.hook_mlp_out and blocks.0.hook_mlp_out: 13.239885330200195
Variance of difference in activations between blocks.1.hook_mlp_out and blocks.0.hook_mlp_out: 2.6880593299865723


In [11]:
df_norms = measure_residual_norms(model, input_seq)
print(df_norms)
#chekpt0

    layer_idx                     hook_name        component       norm
0           0       blocks.0.hook_resid_pre   hook_resid_pre   1.297573
1           0  blocks.0.ln1.hook_normalized  hook_normalized   7.990040
2           0        blocks.0.hook_attn_out    hook_attn_out   3.065934
3           0       blocks.0.hook_resid_mid   hook_resid_mid   3.448415
4           0  blocks.0.ln2.hook_normalized  hook_normalized   8.036119
5           0         blocks.0.mlp.hook_pre         hook_pre  14.750746
6           0        blocks.0.mlp.hook_post        hook_post  11.006336
7           0         blocks.0.hook_mlp_out     hook_mlp_out  10.361373
8           0      blocks.0.hook_resid_post  hook_resid_post  10.660256
9           1       blocks.1.hook_resid_pre   hook_resid_pre  10.660256
10          1  blocks.1.ln1.hook_normalized  hook_normalized   7.998356
11          1        blocks.1.hook_attn_out    hook_attn_out   2.170738
12          1       blocks.1.hook_resid_mid   hook_resid_mid  11

In [20]:
df_norms = measure_residual_norms(model, input_seq)
print(df_norms)
#chekptfinal

    layer_idx                     hook_name        component       norm
0           0       blocks.0.hook_resid_pre   hook_resid_pre   1.117845
1           0  blocks.0.ln1.hook_normalized  hook_normalized   8.000730
2           0        blocks.0.hook_attn_out    hook_attn_out   1.764405
3           0       blocks.0.hook_resid_mid   hook_resid_mid   2.067586
4           0  blocks.0.ln2.hook_normalized  hook_normalized   8.008543
5           0         blocks.0.mlp.hook_pre         hook_pre  12.904223
6           0        blocks.0.mlp.hook_post        hook_post   9.253723
7           0         blocks.0.hook_mlp_out     hook_mlp_out   7.866004
8           0      blocks.0.hook_resid_post  hook_resid_post   8.257267
9           1       blocks.1.hook_resid_pre   hook_resid_pre   8.257267
10          1  blocks.1.ln1.hook_normalized  hook_normalized   8.000715
11          1        blocks.1.hook_attn_out    hook_attn_out   1.840300
12          1       blocks.1.hook_resid_mid   hook_resid_mid   8

In [23]:
df_norms = measure_residual_norms(model, input_seq)
print(df_norms)
#linear model

    layer_idx                     hook_name        component       norm
0           0       blocks.0.hook_resid_pre   hook_resid_pre   1.198434
1           0  blocks.0.ln1.hook_normalized  hook_normalized   7.994504
2           0        blocks.0.hook_attn_out    hook_attn_out   2.269022
3           0       blocks.0.hook_resid_mid   hook_resid_mid   2.591191
4           0  blocks.0.ln2.hook_normalized  hook_normalized   8.032362
5           0         blocks.0.mlp.hook_pre         hook_pre  13.921023
6           0        blocks.0.mlp.hook_post        hook_post  10.077938
7           0         blocks.0.hook_mlp_out     hook_mlp_out   8.696688
8           0      blocks.0.hook_resid_post  hook_resid_post   9.229602
9           1       blocks.1.hook_resid_pre   hook_resid_pre   9.229602
10          1  blocks.1.ln1.hook_normalized  hook_normalized   7.998013
11          1        blocks.1.hook_attn_out    hook_attn_out   1.919553
12          1       blocks.1.hook_resid_mid   hook_resid_mid   9